# Prompting Techniques Ablation (Assignment Evidence)

This notebook compares different LLM prompting techniques on the same held-out prompt set with the same model and decoding settings.

It is designed to provide report-ready evidence for:
- settings/techniques comparison
- schema + safety + relevance + latency metrics
- qualitative examples
- clear conclusion of the best trade-off technique


In [ ]:
# Reproducibility
SEED = 42

import os
import random
import numpy as np
import torch

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Prefer deterministic kernels when supported
if hasattr(torch, "use_deterministic_algorithms"):
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass

if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"SEED set to {SEED}")

## 1) Setup and data loading

This cell loads the same evaluation data used in the project runbook (`data/sft_finbot.eval.jsonl`).


In [1]:
!pip install -qU pandas torch transformers

In [2]:
import json
import re
import time
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
EVAL_PATH = Path("../data/sft_finbot.eval.jsonl")
MAX_SAMPLES = 10

assert EVAL_PATH.exists(), f"Missing eval file: {EVAL_PATH}"

eval_records = []
for line in EVAL_PATH.read_text(encoding="utf-8").splitlines():
    if line.strip():
        eval_records.append(json.loads(line))

eval_records = eval_records[:MAX_SAMPLES]
print(f"Loaded {len(eval_records)} eval records")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=(torch.float16 if torch.cuda.is_available() else torch.float32),
)
model.to(device)
model.eval()


/Users/vyvu/workspaces/mqu/semester3/COMP8420/a2/finbot2/git/financialbot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 10 eval records
Device: cpu


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:06<00:00, 50.07it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

## 2) Prompt techniques under test

We test four techniques using the same user profile context.


In [3]:
SYSTEM_BASE = (
    "You are a cautious financial assistant. "
    "Return ONLY valid JSON with keys: profile_summary, recommendation, reasoning, risks_caveats, sources, disclaimer."
)

FEW_SHOT_EXAMPLE = {
    "user": "Goal: save for emergency fund; Income: 60-120K; Capital: 10-50K; Horizon: SHORT; Risk: LOW",
    "assistant": {
        "profile_summary": "User has moderate income and low risk preference for short horizon.",
        "recommendation": "Prioritize emergency-fund accumulation in high-yield savings and short-term fixed-income.",
        "reasoning": "Because horizon is short and risk tolerance is low, capital preservation dominates return-seeking.",
        "risks_caveats": "Inflation may reduce real returns; reassess every 3 months.",
        "sources": ["User profile fields"],
        "disclaimer": "Educational only, not financial advice."
    }
}

def build_user_text(row):
    profile = row.get("profile", {})
    unknown = row.get("unknown_fields", [])
    return (
        f"Goal: {profile.get('GOAL', 'UNKNOWN')}\n"
        f"Income: {profile.get('INCOME_BAND', 'UNKNOWN')}\n"
        f"Capital: {profile.get('CAPITAL_RANGE', 'UNKNOWN')}\n"
        f"Horizon: {profile.get('TIME_HORIZON', 'UNKNOWN')}\n"
        f"Risk: {profile.get('RISK_TOLERANCE', 'UNKNOWN')}\n"
        f"Unknown fields: {unknown}"
    )

def prompt_baseline(user_text):
    return [
        {"role": "system", "content": SYSTEM_BASE},
        {"role": "user", "content": user_text},
    ]

def prompt_few_shot(user_text):
    return [
        {"role": "system", "content": SYSTEM_BASE + " Follow the style of the demonstration example."},
        {"role": "user", "content": FEW_SHOT_EXAMPLE['user']},
        {"role": "assistant", "content": json.dumps(FEW_SHOT_EXAMPLE['assistant'], ensure_ascii=False)},
        {"role": "user", "content": user_text},
    ]

def prompt_brief_reasoning(user_text):
    return [
        {
            "role": "system",
            "content": SYSTEM_BASE + " Include concise reasoning sentences that link profile fields to each action.",
        },
        {"role": "user", "content": user_text},
    ]

def prompt_self_check(user_text):
    return [
        {
            "role": "system",
            "content": (
                SYSTEM_BASE
                + " First draft answer mentally, then output only final JSON that is schema-valid, specific, and includes caveats."
            ),
        },
        {"role": "user", "content": user_text},
    ]

TECHNIQUES = {
    "baseline_structured": prompt_baseline,
    "few_shot_structured": prompt_few_shot,
    "brief_reasoning": prompt_brief_reasoning,
    "self_check_instruction": prompt_self_check,
}


## 3) Evaluation helpers

Metrics: schema pass, safety pass, relevance score, latency.


In [4]:
BAD_PATTERNS = [
    r"guaranteed\\s+profit",
    r"risk[- ]?free\\s+return",
    r"all[- ]in",
]

def generate_text(messages, max_new_tokens=420):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    dt_ms = (time.time() - t0) * 1000
    text = tokenizer.decode(out[0][enc.input_ids.shape[-1]:], skip_special_tokens=True).strip()
    return text, dt_ms

def parse_json(text):
    try:
        return json.loads(text), True
    except Exception:
        return None, False

def safety_pass(text):
    low = text.lower()
    return not any(re.search(p, low) for p in BAD_PATTERNS)

def relevance_score(payload, row):
    if not isinstance(payload, dict):
        return 0.0
    text = " ".join([
        str(payload.get("profile_summary", "")),
        str(payload.get("recommendation", "")),
        str(payload.get("reasoning", "")),
    ]).lower()
    profile = row.get("profile", {})
    checks = [
        str(profile.get("GOAL", "")).split(" ")[0].lower() in text if profile.get("GOAL") else False,
        str(profile.get("RISK_TOLERANCE", "")).lower() in text if profile.get("RISK_TOLERANCE") else False,
        str(profile.get("TIME_HORIZON", "")).split(" ")[0].lower() in text if profile.get("TIME_HORIZON") else False,
        any(k in text for k in ["allocation", "percentage", "position", "stop-loss", "rebalance"]),
    ]
    return sum(checks) / len(checks)


## 4) Run ablation and build comparison table


In [5]:
rows = []

for idx, row in enumerate(eval_records, start=1):
    user_text = build_user_text(row)
    for name, builder in TECHNIQUES.items():
        messages = builder(user_text)
        text, latency_ms = generate_text(messages)
        payload, schema_ok = parse_json(text)
        s_ok = safety_pass(text)
        rel = relevance_score(payload, row)
        rows.append({
            "idx": idx,
            "technique": name,
            "schema_ok": schema_ok,
            "safety_ok": s_ok,
            "relevance": rel,
            "latency_ms": latency_ms,
            "output": text,
            "user_text": user_text,
        })

df = pd.DataFrame(rows)

summary = (
    df.groupby("technique", as_index=False)
      .agg(
          schema_pass_rate=("schema_ok", "mean"),
          safety_pass_rate=("safety_ok", "mean"),
          relevance_score=("relevance", "mean"),
          mean_latency_ms=("latency_ms", "mean"),
      )
      .sort_values(by=["relevance_score", "schema_pass_rate", "safety_pass_rate"], ascending=False)
)

summary


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

## 5) Qualitative examples (2-3 prompts)


In [ ]:
example_ids = [1, 2, 3]
for ex in example_ids:
    print("=" * 100)
    print(f"PROMPT #{ex}")
    subset = df[df["idx"] == ex]
    if subset.empty:
        continue
    print(subset.iloc[0]["user_text"])
    for _, r in subset.iterrows():
        print("-" * 100)
        print(f"Technique: {r['technique']}")
        print(f"schema_ok={r['schema_ok']} safety_ok={r['safety_ok']} relevance={r['relevance']:.2f} latency_ms={r['latency_ms']:.1f}")
        print(r["output"][:900])


## 6) Short conclusion


In [ ]:
best = summary.iloc[0]
print("Best overall technique:", best["technique"])
print("\nWhy:")
print(f"- Highest relevance score: {best['relevance_score']:.3f}")
print(f"- Schema pass rate: {best['schema_pass_rate']:.3f}")
print(f"- Safety pass rate: {best['safety_pass_rate']:.3f}")
print(f"- Mean latency: {best['mean_latency_ms']:.1f} ms")

print("\nTrade-off notes:")
fastest = summary.sort_values("mean_latency_ms", ascending=True).iloc[0]
print(f"- Fastest technique: {fastest['technique']} ({fastest['mean_latency_ms']:.1f} ms)")
print("- Choose best-overall for quality; choose fastest if latency budget is strict.")

# Optional artifact export for report appendix
summary.to_csv("prompting_techniques_summary.csv", index=False)
df.to_csv("prompting_techniques_raw.csv", index=False)
print("\nSaved: prompting_techniques_summary.csv, prompting_techniques_raw.csv")
